# ⚡ Paralen potek dela agentov z Microsoft Foundry (Python)

## 📋 Napredni vodič za paralelno obdelavo

Ta zvezek prikazuje **vzorce sočasnih potekov dela** z uporabo Microsoft Agent Framework. Naučili se boste, kako ustvariti visoko zmogljive poteze dela z vzporedno obdelavo, kjer več AI agentov izvaja istočasno, kar dramatično izboljša prepustnost in omogoča sofisticirane večnitočne poslovne procese.

> **Opomba o migraciji:** Ta primer je prej uporabljal GitHub Modelle. GitHub Modelle so ukinjeni (upokojitev julij 2026), zato zdaj uporablja **Microsoft Foundry** preko `FoundryChatClient`, ki cilja na Azure OpenAI **Responses API**.

## 🎯 Cilji učenja

### 🚀 **Osnove sočasne obdelave**
- **Vzporedno izvajanje agentov**: Zaženite več agentov istočasno za maksimalno učinkovitost
- **Orkestracija potekov dela**: Koordinirajte sočasne operacije ob ohranjanju doslednosti podatkov
- **Optimizacija zmogljivosti**: Dosezite znatno pospeševanje skozi paralelno obdelavo
- **Upravljanje virov**: Učinkovita raba virov AI modelov med sočasnimi operacijami

### 🏗️ **Napredni vzorci sočasnosti**
- **Fork-Join obdelava**: Razdelitev dela med več agentov in združevanje rezultatov
- **Paralelizem cevovoda**: Prekrivajoče se izvedbene faze za neprekinjeno prepustnost
- **Uravnoteženje obremenitve**: Enakomerno razporejanje dela med razpoložljive vire agentov
- **Točke sinhronizacije**: Koordinacija sočasnih agentov v kritičnih fazah poteka dela

### 🏢 **Sočasne poslovne aplikacije**
- **Obdelava velikih količin dokumentov**: Istočasna obdelava več dokumentov
- **Analiza vsebine v realnem času**: Sočasna analiza prihajajočih podatkovnih tokov
- **Optimizacija serijske obdelave**: Makismalna prepustnost za obsežne operacije
- **Večmodalna analiza**: Paralelna obdelava različnih vrst vsebin (besedilo, slike, podatki)

## ⚙️ Zahteve in nastavitev

### 📦 **Zahtevane odvisnosti**

Namestite Agent Framework z zmožnostmi sočasnih potekov dela:

```bash
pip install agent-framework -U
```

### 🔑 **Konfiguracija Microsoft Foundry**

Prijavite se z Azure CLI (`az login`), da se lahko `AzureCliCredential` avtorizira, nato nastavite podatke o vašem Microsoft Foundry projektu.

**Nastavitev okolja (.env datoteka):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

**Razmisleki o sočasni obdelavi:**
- **Omejitve hitrosti**: Spremljajte omejitve Azure OpenAI za sočasne zahteve
- **Poraba virov**: Upoštevajte porabo pomnilnika in CPU pri več sočasnih agentih
- **Ravnanje z napakami**: Implementirajte robustno obnovo napak pri paralelnih operacijah

### 🏗️ **Arhitektura sočasnih potekov dela**

```mermaid
graph TD
    A[Začetek poteka dela] --> B[Sočasno izvajanje]
    B --> C[Skupina agentov 1]
    B --> D[Skupina agentov 2]
    B --> E[Skupina agentov 3]
    C --> F[Združevanje rezultatov]
    D --> F
    E --> F
    F --> G[Končni izhod]
    
    H[Microsoft Foundry] --> C
    H --> D
    H --> E
```

**Ključne prednosti:**
- **⚡ Zmogljivost**: Znatno pospeševanje skozi paralelno izvajanje
- **📈 Razširljivost**: Obvladovanje povečane obremenitve brez sorazmernega podaljšanja časa
- **🔄 Učinkovitost**: Boljša uporaba razpoložljivih računalniških virov
- **🎯 Prepustnost**: Obdelajte več dela v enakem času

## 🎨 **Oblika vzorcev sočasnih potekov dela**

### 🔍 **Cevovod raziskav in analiz**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Potek obdelave podatkov**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Cevovod ustvarjanja vsebine**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Večstopenjska obdelava**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Poslovne koristi zmogljivosti**

### ⚡ **Optimizacija prepustnosti**
- **Paralelno izvajanje**: Več agentov dela istočasno
- **Izraba virov**: Maksimalna učinkovitost razpoložljive kapacitete AI modela
- **Zmanjšanje časa**: Znatno zmanjšanje skupnega časa obdelave
- **Razširljiva arhitektura**: Enostavno dodajanje več sočasnih agentov po potrebi

### 🛡️ **Zanesljivost in odpornost**
- **Otpornost na napake**: Napake posameznih agentov ne ustavijo celotnega poteka dela
- **Izolacija napak**: Težave v eni sočasni vejo ne vplivajo na druge
- **Vljudna degradacija**: Sistem deluje tudi z zmanjšano zmogljivostjo agentov
- **Mehanizmi obnove**: Samodejno ponavljanje in obravnava napak pri neuspešnih operacijah

### 📊 **Nadzor in opazovanje**
- **Sledenje sočasnemu izvajanju**: Spremljajte napredek vseh paralelnih operacij
- **Metrične zmogljivosti**: Merite pospeševanje in izboljšave učinkovitosti
- **Analiza porabe virov**: Optimizirajte dodeljevanje sočasnih agentov
- **Prepoznavanje ozkih grl**: Poiščite in odpravite omejitve zmogljivosti

Zgradimo visoko zmogljive sočasne AI poteke dela! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = provider.as_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = provider.as_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)


In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
